# Notebook 4: Training Pipeline trên Google Colab (Phase 3)

**Mục đích:** Train end-to-end pipeline Faster R-CNN (+ GAT) trên COCO, có:
- COCO DataLoader (TorchVision `CocoDetection`)
- Adam optimizer + LR schedule **1e-4 → 1e-5 (@epoch 8) → 1e-6 (@epoch 11)**
- **Frozen backbone 2 epoch đầu**, unfreeze từ epoch 3
- **Checkpoint mỗi epoch** (lưu lên Google Drive)
- **mAP evaluation** bằng `pycocotools` sau mỗi epoch

> Thiết kế cho **Colab T4/A100**. Logic train/eval nằm trong `src/trainer.py` và
> `src/evaluator.py` — notebook này chỉ lắp ráp và chạy.

**Cấu hình khuyến nghị (CONTEXT.md):** batch size 2-4 cho RTX 3050 (4GB), 8 cho A100.

---
## Outline
0. Kiểm tra môi trường + cài `pycocotools`
1. Lấy code `src/` + mount Drive (checkpoint)
2. Tải COCO (demo dùng val2017 cho nhanh)
3. Dataset + DataLoader
4. Build model
5. Training loop (12 epoch) + eval mAP mỗi epoch
6. Vẽ loss & mAP
7. Load checkpoint + inference thử

## Bước 0: Kiểm tra môi trường + cài pycocotools

Colab đã có sẵn `torch`/`torchvision`. Chỉ cần thêm `pycocotools` cho mAP.

In [ ]:
import sys, subprocess

# cài pycocotools nếu chưa có (chạy được cả Colab lẫn local)
try:
    import pycocotools  # noqa: F401
    print("pycocotools đã có sẵn.")
except ImportError:
    print("Đang cài pycocotools...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools"],
                   check=True)
    print("Xong.")

import torch, torchvision
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | torchvision {torchvision.__version__}")
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)} | "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## Bước 1: Clone code từ GitHub + mount Google Drive

Clone repo của bạn về Colab để lấy `src/` (gat_module, context_classifier, trainer, evaluator),
rồi mount Drive để lưu checkpoint (không mất khi Colab ngắt phiên).

> **Sửa `REPO_URL`** thành link repo của bạn ở cell dưới. Repo phải **public**
> (hoặc dùng token nếu private). Cell sẽ tự tìm thư mục `src` bất kể tên repo,
> và `git pull` nếu repo đã clone từ trước (tiện khi bạn push code mới rồi chạy lại).

In [ ]:
import os, sys, glob

# Repo của bạn (public) — sửa ở đây nếu đổi repo
REPO_URL = "https://github.com/PwiseV/SE356-GNN-Computer-Vision.git"

repo_name = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
repo_dir = os.path.join('/content', repo_name)

if not os.path.isdir(repo_dir):
    print(f"Cloning {REPO_URL} ...")
    rc = os.system(f"git clone {REPO_URL} {repo_dir}")
    assert rc == 0, "git clone thất bại — kiểm tra URL và repo có public không?"
else:
    print("Repo đã có sẵn → git pull bản mới nhất...")
    os.system(f"git -C {repo_dir} pull")

# tự tìm thư mục src (chứa trainer.py) bất kể cấu trúc repo
hits = glob.glob(repo_dir + "/**/trainer.py", recursive=True)
assert hits, f"Không thấy trainer.py trong {repo_dir} — repo đã có src/ chưa?"
SRC_DIR = os.path.dirname(hits[0])
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)
print("SRC_DIR:", SRC_DIR)

# mount Google Drive cho checkpoint (chỉ chạy trên Colab)
CKPT_DIR = "checkpoints"
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = "/content/drive/MyDrive/gat_checkpoints"
except Exception:
    print("Không mount Drive → checkpoint lưu local (mất khi ngắt phiên):", CKPT_DIR)
os.makedirs(CKPT_DIR, exist_ok=True)
print("CKPT_DIR:", CKPT_DIR)

# import code từ src
from gat_module import GAT
from context_classifier import ContentModerationGAT
import trainer
from evaluator import evaluate_coco
print("✓ Import src OK:", trainer.__file__)

## Bước 2: Tải COCO dataset

Demo dùng **val2017** (5000 ảnh, ~780MB) làm tập train cho nhanh. Khi train thật
hãy đổi sang **train2017** (118K ảnh, ~19GB). Annotation dùng `instances_*2017.json`.

> Issue SSL (CONTEXT.md) đã xử lý bằng `ssl._create_unverified_context`.

In [ ]:
import os, ssl, zipfile, urllib.request

ssl._create_default_https_context = ssl._create_unverified_context

DATA_DIR = os.path.abspath("coco")          # đường dẫn tuyệt đối, tránh lệch cwd
os.makedirs(DATA_DIR, exist_ok=True)
IMG_DIR = os.path.join(DATA_DIR, "val2017")
ANN_FILE = os.path.join(DATA_DIR, "annotations", "instances_val2017.json")

# mỗi job: (tên zip, url, marker đích để biết đã có chưa)
JOBS = [
    ("val2017.zip",
     "http://images.cocodataset.org/zips/val2017.zip", IMG_DIR),
    ("annotations_trainval2017.zip",
     "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
     ANN_FILE),
]

def have(marker):
    """True nếu marker là file tồn tại, hoặc thư mục không rỗng."""
    if os.path.isdir(marker):
        return len(os.listdir(marker)) > 0
    return os.path.isfile(marker)

for name, url, marker in JOBS:
    if have(marker):
        print(f"  ✓ đã có: {marker}")
        continue
    zpath = os.path.join(DATA_DIR, name)
    print(f"  tải {name} (có thể vài phút)...")
    urllib.request.urlretrieve(url, zpath)
    print(f"  giải nén {name} ...")
    with zipfile.ZipFile(zpath) as z:
        z.extractall(DATA_DIR)
    os.remove(zpath)
    # KIỂM TRA marker chính xác (file json / thư mục ảnh), không chỉ check thư mục cha
    assert have(marker), f"Giải nén xong nhưng vẫn không thấy {marker}"
    print(f"  ✓ {marker}")

# chốt: cả ảnh lẫn annotation phải tồn tại mới sang Bước 3 được
assert os.path.isdir(IMG_DIR), f"THIẾU ảnh: {IMG_DIR}"
assert os.path.isfile(ANN_FILE), f"THIẾU annotation: {ANN_FILE}"
print(f"\n✅ COCO sẵn sàng | {len(os.listdir(IMG_DIR))} ảnh val2017")
print("IMG_DIR  =", IMG_DIR)
print("ANN_FILE =", ANN_FILE)

## Bước 3: Dataset + DataLoader

`CocoDetectionDataset` (trong `trainer.py`) bọc TorchVision `CocoDetection` →
trả về `(image_tensor, target_dict)` đúng định dạng Faster R-CNN, kèm `image_id`
cho evaluator.

Demo dùng chung val2017 cho cả train & val (chỉ để pipeline chạy). Khi train thật:
train = train2017, val = val2017.

In [ ]:
import os
from trainer import CocoDetectionDataset, make_loader

# chặn lỗi FileNotFoundError: đảm bảo Bước 2 đã tải xong COCO
assert os.path.isdir(IMG_DIR) and os.path.isfile(ANN_FILE), (
    "Chưa thấy COCO. Hãy chạy lại BƯỚC 2 tới khi in '✅ COCO sẵn sàng', "
    "rồi mới chạy cell này.")

# batch size: 2-4 cho GPU 4GB, 8 cho A100 (CONTEXT.md)
BATCH_SIZE = 4

dataset = CocoDetectionDataset(IMG_DIR, ANN_FILE)
print(f"Số ảnh: {len(dataset)}")

# demo: train & val đều trỏ vào val2017 (thật thì tách 2 tập)
train_loader = make_loader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = make_loader(dataset, batch_size=1, shuffle=False, num_workers=2)

# xem thử 1 sample
img, target = dataset[0]
print(f"image tensor: {tuple(img.shape)} | boxes: {tuple(target['boxes'].shape)} | "
      f"labels: {target['labels'].tolist()[:5]}...")

## Bước 4: Build model

Demo fine-tune từ Faster R-CNN **pretrained COCO** (hội tụ nhanh, hợp Colab).

**Vị trí GAT plug-in:** `ContentModerationGAT` (Phase 2) bọc detector này và thêm
GAT + Safety Head. Train end-to-end qua RoI head có GAT là phần nghiên cứu mở rộng;
ở Phase 3 ta train detector chuẩn để có baseline mAP + checkpoint, rồi gắn GAT lên trên.

In [ ]:
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights)

# pretrained COCO để fine-tune (demo). Train from scratch: weights=None
model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: Faster R-CNN ResNet50-FPN | params: {n_params:,}")

# (tuỳ chọn) bọc GAT lên trên để dùng ở inference/explainability:
# cm = ContentModerationGAT(build_detector=False)
# cm.detector = model  # tái dùng detector vừa load

## Bước 5: Training loop + eval mAP mỗi epoch

`trainer.train(...)` lo toàn bộ: frozen backbone 2 epoch đầu → unfreeze → train →
`scheduler.step()` → checkpoint → eval mAP.

- `num_epochs=12`, LR schedule milestones `(8, 11)`.
- `eval_fn` giới hạn `max_images` để eval nhanh trong demo (bỏ giới hạn khi train thật).

> ⏱️ Trên Colab T4, 1 epoch val2017 mất vài phút; full train2017 lâu hơn nhiều.
> Để chạy thử nhanh, hạ `num_epochs` xuống 1-2.

In [ ]:
# eval nhanh trong demo: chỉ 200 ảnh đầu. Train thật: bỏ max_images.
def eval_fn(m, dl, dev):
    return evaluate_coco(m, dl, dev, max_images=200)

history = trainer.train(
    model,
    train_loader=train_loader,
    device=device,
    num_epochs=1,                  # smoke test 1 epoch. Train thật: tăng lên 12
    lr=1e-4,
    milestones=(8, 11),            # 1e-4 → 1e-5 → 1e-6 (chỉ có tác dụng khi >=12 epoch)
    gamma=0.1,
    freeze_backbone_epochs=2,      # frozen 2 epoch đầu
    checkpoint_dir=CKPT_DIR,
    evaluate_fn=eval_fn,
    val_loader=val_loader,
    print_freq=50,
)
print("\nDone. history keys:", list(history.keys()))

## Bước 6: Vẽ training loss & mAP theo epoch

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history['train_loss'], marker='o')
axes[0].set_title('Training loss'); axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
axes[0].grid(alpha=0.3)

if history['mAP']:
    axes[1].plot(history['mAP'], marker='s', color='green')
    axes[1].set_title('Validation mAP (COCO)'); axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('mAP @0.5:0.95'); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## Bước 7: Load checkpoint + inference thử

Nạp checkpoint epoch cuối rồi detect trên 1 ảnh val để kiểm tra.

In [ ]:
import glob
from trainer import load_checkpoint

ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "checkpoint_epoch_*.pth")))
print("Checkpoints:", [os.path.basename(c) for c in ckpts])

if ckpts:
    ep = load_checkpoint(ckpts[-1], model, map_location=device)
    print(f"Đã nạp checkpoint epoch {ep}")

    model.eval()
    img, target = dataset[0]
    with torch.no_grad():
        pred = model([img.to(device)])[0]
    keep = pred['scores'] > 0.5
    print(f"Detect được {int(keep.sum())} object (score>0.5) trên ảnh val đầu tiên.")
    print("Labels:", pred['labels'][keep].cpu().tolist())

---
## 🎓 Tổng kết Notebook 4 (Phase 3)

Đã có **training pipeline đầy đủ, chạy được trên Colab**:
1. ✅ COCO DataLoader (`CocoDetectionDataset` + `collate_fn`)
2. ✅ Adam + LR schedule 1e-4 → 1e-5 (@8) → 1e-6 (@11)
3. ✅ Frozen backbone 2 epoch đầu → unfreeze epoch 3
4. ✅ Checkpoint mỗi epoch (lưu Google Drive)
5. ✅ mAP evaluation bằng pycocotools sau mỗi epoch
6. ✅ Logic tách ra `src/trainer.py` + `src/evaluator.py` (đã unit-test)

### Lưu ý khi train thật
- Đổi `IMG_DIR`/`ANN_FILE` sang **train2017** cho tập train, giữ val2017 cho eval.
- Batch size: 8 (A100) / 2-4 (T4 hoặc RTX 3050 4GB) + `torch.cuda.empty_cache()` (đã có trong trainer).
- Bỏ `max_images` trong `eval_fn` để tính mAP trên toàn val set.
- Full 12 epoch trên train2017 mất nhiều giờ → dùng A100 + để Colab chạy nền.

### Tiếp theo (Phase 4)
- `src/visualizer.py`: `draw_attention_graph`, `draw_safety_overlay`, `generate_report`
- Train Safety Head của `ContentModerationGAT` trên nhãn safe/unsafe (tận dụng pipeline này).